# Powderday flux catalogs — quenched galaxies in the high-res 25 Mpc box

**Goal.** Multi-aperture, dusty vs dust-free photometric catalogs (fluxes **with errors**) for
**quenched** galaxies in SIMBA high-res **m25n512** (`cis25`) at **z ≈ 0.3, 0.6, 0.7, 1.0, 2.0**.

**Sample (per anchor snapshot).** `log10 M* > 10`, **passive** by the 0.2/τ criterion
(sSFR < 0.2/t_H at the anchor), and **> 20 gas particles** (plus the usual ≥ 20 star-particle floor).
The sample is split by **weak vs strong AGN feedback over the quench window**: the AGN–ISM coupling
strength `xcoup_hist` (jet-mode strength gated by gas-poorness, §8j physics) averaged between each
galaxy's **SFT and QT** (1/t and 0.2/t crossings from `find_quenching_times`), tercile split.

**Pipeline** (same skeleton as `test_powderday.ipynb`, selection machinery from
`quench_mode_vs_sigma_gas.ipynb`):

| Part | What | Where |
|---|---|---|
| 1 | anchors + gated `BUILD_MULTI_Z` / `BUILD_BH` history builds | cluster |
| 2–3 | selection, SFT/QT, AGN split, **sample statistics** | anywhere (needs the HDF5s) |
| 4 | Stage 0 — per-galaxy particle files | cluster |
| 5 | Stage 1 — selection HDF5 + Slurm masters (dust_on / dust_off) → run RT | cluster |
| 6 | aperture QC on the first `.rtout.sed` | cluster |
| 7 | Stage 2 — per-aperture flux extraction → **one catalog per aperture per dust mode** | cluster |

**Apertures.** Hyperion log-spaces SED apertures between `ap_min` and `ap_max`
(`set_aperture_range`), so exactly {10, 20, 30} kpc is not representable; we use the factor-2
ladder **10, 20, 40, 80, 160 kpc** (5 apertures; 160 kpc ≈ total for a plist particle cutout).
Change `N_AP / AP_MIN_KPC / AP_MAX_KPC` below (they must match the values in the parameter
masters that the RT jobs copy). **Requires the one-time powderday patch documented before Stage 1.**

**Flux errors.** Hyperion's Monte-Carlo SED uncertainty, read with
`get_sed(..., uncertainties=True)` and propagated through the filter convolution
(`<filter>_err` columns; NaN if a run stored no uncertainties).

# Part 0 — Setup & configuration

In [ ]:
import os
import gc
import glob
import json
import warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.table import Table, vstack, join
from astropy import units as u
from astropy.cosmology import Planck15 as COSMO   # matches the quenching machinery

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory, caesar_read_progen
from simbanator.analysis.quenching import find_quenching_times

# ── simulation ────────────────────────────────────────────────────────────────
SIM_NAME = "cis25"        # SIMBA high-res 25 Mpc/h box (m25n512); must exist in ~/.simbanator/config.json
try:
    sim = Simulation(SIM_NAME)
except KeyError as e:
    raise KeyError(
        f"'{SIM_NAME}' is not registered in ~/.simbanator/config.json on this machine.\n"
        "Register it once (adjust paths to where the 25 Mpc snapshots+catalogs live):\n"
        "  from simbanator.io.config import add_simulation\n"
        "  add_simulation('cis25', data_dir='<...>/SIMBA_25/s25',\n"
        "                 catalog_dir='<...>/SIMBA_25/s25/Groups',\n"
        "                 file_format='m25n512_{snap:03d}.hdf5')\n"
        "then add \"snap_z_map\": \"zsnap_map_caesar_box100.txt\" to that entry "
        "(SIMBA boxes share the snapshot schedule)."
    ) from e
if sim.scale_factors is None:
    raise ValueError(f"'{SIM_NAME}' config has no snap_z_map — add "
                     '"snap_z_map": "zsnap_map_caesar_box100.txt" to its entry in ~/.simbanator/config.json')

# filtered-particle filename prefix (Stage 0 == Stage 1, never let them drift)
PARTICLE_PREFIX = sim.file_format.split("_{")[0]        # 'm25n512'

# ── selection: quenched + massive + realistically gas-populated ───────────────
TARGET_REDSHIFTS = [0.3, 0.6, 0.7, 1.0, 2.0]
MASS_FLOOR       = 10.0        # log10(M*/Msun) > 10
PASSIVE_FACTOR   = 0.2         # passive if sSFR < 0.2 / t_H  (== the QT threshold of find_quenching_times)
NGAS_MIN         = 21          # STRICTLY > 20 gas particles at the anchor
NSTAR_MIN        = 20          # star-particle floor (same as quench_mode_vs_sigma_gas)

# ── AGN / coupling constants (identical to quench_mode_vs_sigma_gas §0) ──────
JET_LOGMBH    = 7.5            # jet mode: log10(M_BH) > 7.5 ...
JET_FEDD      = 0.2            #           ... AND f_Edd < 0.2
XRAY_FEDD_MAX = 0.02           # (kept for reference; xcoup uses the f_gas gate)
XRAY_FGAS_MAX = 0.2            # coupling gate: f_gas = Mgas/M* < 0.2
GYR = 1e9

# ── history tracking ──────────────────────────────────────────────────────────
TRACK_AGE_FRAC      = 0.09     # track back to ~this fraction of the cosmic age at selection
ANCHOR_END_OVERRIDE = {}
CORRUPT_SNAPS       = set()

# ── heavy-build gates (set True on the cluster, then reuse the cached HDF5s) ──
BUILD_MULTI_Z = False          # per-anchor progenitor FITS + property history HDF5
BUILD_BH      = False          # per-anchor BH (mass / mdot / f_Edd) history HDF5

# ── apertures (MUST match SED_APERTURE_* in simbanator/sed/parameters_master*.py) ──
N_AP       = 5
AP_MIN_KPC = 10.0
AP_MAX_KPC = 160.0
APERTURE_RADII_KPC = np.geomspace(AP_MIN_KPC, AP_MAX_KPC, N_AP)   # 10, 20, 40, 80, 160
APERTURE_LABELS    = [f"ap{r:g}kpc" for r in APERTURE_RADII_KPC]
APERTURE_LABELS[-1] += "_total"          # the largest aperture ≈ total flux of the cutout

# ── powderday run layout (same conventions as test_powderday.ipynb) ──────────
GVFS_BASE   = ''
# '+' not os.path.join: with GVFS_BASE='' this must stay ABSOLUTE (see test_powderday)
REMOTE_HOME = GVFS_BASE + "/mnt/home/glorenzon/analize_simba_cgm"

hydro_dir_base = os.path.join(os.getcwd(), 'output', sim.name, 'filtered_particles')
selection_file = 'selection_m25_quenched'                  # MakeSED appends '.h5'
sed_output_dir = os.path.join(REMOTE_HOME, 'output', sim.name, 'sed_quenched_apertures')

RUNS = {
    'dust_on':  dict(run_tag='dusty_simdust', paramf='parameters_master.py'),
    'dust_off': dict(run_tag='nodust_1e-12',  paramf='parameters_master-nodust.py'),
}

# ── local output tree ─────────────────────────────────────────────────────────
OUT      = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR   = os.path.join(OUT, "caesar_sfh")
TABLEDIR = os.path.join(OUT, "tables")
PLOTDIR  = os.path.join(OUT, "plots", "powderday_quenched")
CATDIR   = os.path.join(OUT, "sed_aperture_catalogs")
for _d in (SFHDIR, TABLEDIR, PLOTDIR, CATDIR):
    os.makedirs(_d, exist_ok=True)
SELECTION_FITS = os.path.join(TABLEDIR, "powderday_quenched_selection.fits")

def _ztag(z):
    return ("z%g" % z).replace(".", "p")

print(f"sim={sim.name}  data_dir={sim.data_dir}")
print(f"prefix={PARTICLE_PREFIX}  anchors z={TARGET_REDSHIFTS}")
print("apertures [kpc]:", np.round(APERTURE_RADII_KPC, 2), "->", APERTURE_LABELS)
print("SED output:", sed_output_dir)

# Part 1 — Anchors & gated cluster builds

Each anchor (z ≈ 0.3, 0.6, 0.7, 1.0, 2.0 → nearest snapshot) gets its **own** progenitor table +
property history with that snapshot as row 0, and a BH history aligned to the same rows — exactly
the `quench_mode_vs_sigma_gas.ipynb` machinery, pointed at `cis25`. Histories are pre-selected to
**massive + passive** at the anchor (the gas/star floors are applied later so the statistics can
count them).

In [ ]:
# ── anchor table: snapshot, track end, per-anchor product paths ──
_sall, _zall = [], []
for _s in range(0, 152):
    try:
        _zv = float(sim.get_z_from_snap(_s))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)
_aall = COSMO.age(_zall).value

ANCHORS = {}
for _zt in TARGET_REDSHIFTS:
    _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
    _age_end = TRACK_AGE_FRAC * float(_aall[_sall == _snap][0])
    _end = int(ANCHOR_END_OVERRIDE.get(_zt, int(_sall[np.searchsorted(_aall, _age_end)])))
    _tag = _ztag(_zt)
    ANCHORS[_zt] = dict(z_target=_zt, tag=_tag, snap=_snap,
                        z=float(sim.get_z_from_snap(_snap)), end_snap=_end,
                        prog_file=f"progenitors_anchor_{_tag}.fits",
                        hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"),
                        bh_path=os.path.join(SFHDIR, f"bh_history_anchor_{_tag}.hdf5"))

print(f"{'z_tgt':>6s} {'snap':>5s} {'z':>7s} {'end':>5s} {'hist':>6s} {'BH':>4s}")
for _zt, A in ANCHORS.items():
    print(f"{_zt:6.1f} {A['snap']:5d} {A['z']:7.3f} {A['end_snap']:5d} "
          f"{'ok' if os.path.exists(A['hist_path']) else '--':>6s} "
          f"{'ok' if os.path.exists(A['bh_path']) else '--':>4s}")

In [ ]:
# ── property list tracked per anchor (superset of what selection + coupling need) ──
PROPS = {
    "galaxy_data": [
        "masses.stellar", "sfr", "masses.gas", "masses.dust", "masses.H2", "masses.HI",
        "radii.stellar_half_mass", "radii.gas_half_mass",
        "pos", "ngas", "nstar", "ages.mass_weighted",
    ],
    "halo_data": ["masses.total"],
}

# ── GATED (cluster): per-anchor progenitor table + property history ──
# Verbatim port of quench_mode_vs_sigma_gas 1z·build, with the (stricter) M*>10 pre-selection.
if BUILD_MULTI_Z:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['hist_path'])}"); continue
        end = int(A["end_snap"])
        while end < A["snap"] and (end in CORRUPT_SNAPS or not os.path.exists(sim.get_caesar_file(end))):
            end += 1
        A["end_snap"] = end
        print(f"[{A['tag']}] anchor snap {A['snap']} (z={A['z']:.2f}) <- {end}: progenitor table ...")
        cs_a = sim.load_catalog(snap=A["snap"])
        caesar_read_progen([g.GroupID for g in cs_a.galaxies], A["prog_file"],
                           range(end, A["snap"] + 1), sim, output_dir=None)
        hist = HDF5BuildHistory(sim, cs_a, progfilename=A["prog_file"])
        with fits.open(hist.progen_file) as hdul:
            valid_ids = np.asarray(hdul[1].data["GroupID"])
            _tHa = COSMO.age(float(A["z"])).value * 1e9
            _gid = np.array([g.GroupID for g in cs_a.galaxies])
            _ms  = np.array([float(g.masses["stellar"]) for g in cs_a.galaxies])
            _sf  = np.array([float(g.sfr) for g in cs_a.galaxies])
            with np.errstate(all="ignore"):
                _ss = np.where(_ms > 0, _sf / _ms, np.nan)
                _ok = (np.log10(np.where(_ms > 0, _ms, np.nan)) > MASS_FLOOR) & (_ss < PASSIVE_FACTOR / _tHa)
            _keep = {int(g) for g in _gid[_ok]}
            valid_ids = np.asarray([i for i in valid_ids if int(i) in _keep], dtype=valid_ids.dtype)
            print(f"  [pre-select] {len(valid_ids)}/{len(_gid)} massive+passive at z={A['z']:.2f}")
        hist.get_history_indx(valid_ids, A["snap"], end)
        props_try = {k: list(v) for k, v in PROPS.items()}
        while True:   # drop-and-retry: some catalog versions miss some fields
            try:
                hist.get_property_history(props_try, verbose=0); break
            except KeyError as e:
                msg = str(e); dropped = False
                for fam, plist in props_try.items():
                    for pr in list(plist):
                        if pr in msg or pr.split("/")[-1] in msg:
                            plist.remove(pr); print("  [drop]", pr); dropped = True
                if not dropped:
                    raise
        hist.save_history_to_hdf5(os.path.basename(A["hist_path"]))
        del cs_a, hist; gc.collect()
        print(f"[{A['tag']}] history -> {A['hist_path']}")
else:
    print("BUILD_MULTI_Z=False -> expecting per-anchor histories under", SFHDIR)

In [ ]:
# ── loaders (verbatim from quench_mode_vs_sigma_gas): row 0 = the anchor epoch ──
def load_anchor_history(A):
    """Load one anchor's history -> dict(galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P)."""
    H = {"P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H

def build_prog_index(A, galaxy_ids, snaps_arr):
    """(n_snap, n_gal) catalogue group-index matrix aligned to the anchor history rows."""
    cs0 = sim.load_catalog(snap=A["snap"])
    hP = HDF5BuildHistory(sim, cs0, progfilename=A["prog_file"])
    hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    M = np.vstack([hP.history_indx[str(s)] for s in snaps_arr])
    del cs0, hP; gc.collect()
    return M

In [ ]:
# ── BH history: per-anchor build (GATED) + loader (verbatim quench_mode §4b) ──
BH_CANDIDATES = {"bh_mass": ["masses.bh", "masses.bh_mass", "bhmass"],
                 "bh_mdot": ["bhmdot", "bh_mdot"],
                 "bh_fedd": ["bh_fedd", "bhfedd", "fedd"]}

def _resolve_bh_path(f, cands):
    for c in cands:
        for p in (f"galaxy_data/dicts/{c}", f"galaxy_data/{c}"):
            if p in f:
                return p
    return None

def build_bh_for_anchor(A, galaxy_ids, snaps_arr, n_gal):
    pidx = build_prog_index(A, galaxy_ids, snaps_arr)
    n_snap = len(snaps_arr)
    BH = {k: np.full((n_snap, n_gal), np.nan) for k in BH_CANDIDATES}
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                valid = np.isfinite(pidx[ri]); cv = np.where(valid)[0]
                vi = pidx[ri][valid].astype(int)
                for k, cands in BH_CANDIDATES.items():
                    p = _resolve_bh_path(f, cands)
                    if p is not None:
                        BH[k][ri, cv] = f[p][:][vi]
        except (OSError, KeyError) as e:
            print(f"  [skip] snap {snap}: {type(e).__name__}"); CORRUPT_SNAPS.add(snap)
    with h5py.File(A["bh_path"], "w") as f:
        for k, arr in BH.items():
            f.create_dataset(k, data=arr)
    print(f"[{A['tag']}] BH history -> {A['bh_path']}")
    return BH

def load_bh(bh_hist_path):
    with h5py.File(bh_hist_path, "r") as f:
        return {k: f[k][:] for k in f.keys()}

if BUILD_BH:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["bh_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['bh_path'])}"); continue
        if not os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] no history yet -> run BUILD_MULTI_Z first"); continue
        _H = load_anchor_history(A)
        build_bh_for_anchor(A, _H["galaxy_ids"], _H["snaps_arr"], len(_H["galaxy_ids"]))
        del _H; gc.collect()
else:
    print("BUILD_BH=False -> expecting per-anchor BH histories under", SFHDIR)

# Part 2 — Selection, quench events (SFT/QT) & the weak/strong AGN split

- **Selection** (at row 0 = the anchor): `log10 M* > 10`, passive (`sSFR < 0.2/t_H`), `ngas > 20`,
  `nstar ≥ 20`.
- **SFT/QT** per galaxy from `find_quenching_times` on the tracked sSFR history (SFT = crossing
  below 1/t, QT = subsequent crossing below 0.2/t with persistence); the **last** event is kept.
- **AGN split**: `xstr_quench` = mean of `xcoup_hist` (jet strength `clip(log10(0.2/f_Edd),0,1)`
  for `log M_BH > 7.5`, gated by `f_gas < 0.2`) over snapshots with `t_SFT ≤ t ≤ t_QT`; if the
  window is narrower than the snapshot spacing, the finite snapshot nearest SFT is used.
  **strong / weak = top / bottom terciles** of `xstr_quench` (per anchor); middle tercile =
  `intermediate`; no finite coupling = `no_AGN`; no detected quench event = `no_event`.

In [ ]:
# ── selection mask at the anchor epoch (row 0) ──
def selection_mask(P, t_cosmic_yr):
    mstar0 = P["masses.stellar"][0]
    sfr0   = P["sfr"][0]
    ngas0  = P["ngas"][0]
    nstar0 = P["nstar"][0] if "nstar" in P else np.full_like(mstar0, np.inf)
    with np.errstate(all="ignore"):
        ssfr0 = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)
        cuts = {
            "massive":  np.log10(np.where(mstar0 > 0, mstar0, np.nan)) > MASS_FLOOR,
            "passive":  ssfr0 < (PASSIVE_FACTOR / t_cosmic_yr[0]),
            "gas>20":   ngas0 >= NGAS_MIN,
            "star>=20": nstar0 >= NSTAR_MIN,
        }
    m = cuts["massive"] & cuts["passive"] & cuts["gas>20"] & cuts["star>=20"]
    return m, cuts

# ── SFT/QT per selected galaxy (trimmed from quench_mode build_records) ──
def quench_records(P, t_cosmic_yr, redshift, galaxy_ids, cols):
    """One record per selected column; galaxies without a detected quench event keep NaN times."""
    records = []
    for col in np.asarray(cols, int):
        gid = galaxy_ids[col]
        mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
        with np.errstate(all="ignore"):
            ssfr = np.where(mstar > 0, sfr / mstar, np.nan)
        valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
        rec = dict(gid=int(gid), col=int(col), t_sft=np.nan, t_qt=np.nan,
                   tau_q=np.nan, tau_q_over_tH=np.nan, z_qt=np.nan)
        if valid.sum() >= 5:
            t = t_cosmic_yr[valid]; s = ssfr[valid]
            o = np.argsort(t); t, s = t[o], s[o]
            tu, ui = np.unique(t, return_index=True); su = s[ui]
            if len(tu) >= 5:
                qts, sfts, _, dbg = find_quenching_times(
                    tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True)
                if len(qts):
                    k = int(np.argmax(qts))                     # last (surviving) quench event
                    rec["t_qt"], rec["t_sft"] = float(qts[k]), float(sfts[k])
                    rec["tau_q"] = rec["t_qt"] - rec["t_sft"]
                    z_qt = float(np.interp(rec["t_qt"], t_cosmic_yr[::-1], redshift[::-1]))
                    rec["z_qt"] = z_qt
                    rec["tau_q_over_tH"] = rec["tau_q"] / (COSMO.age(z_qt).value * 1e9)
        records.append(rec)
    return records

In [ ]:
# ── AGN–ISM coupling over the quench window [SFT, QT] (physics verbatim from §8j build_coupling) ──
def coupling_quench_window(BH, P, records, t_cosmic_yr):
    _ord = np.argsort(t_cosmic_yr); t_inc = t_cosmic_yr[_ord]
    with np.errstate(all="ignore"):
        fgas_hist = np.where(P["masses.stellar"] > 0, P["masses.gas"] / P["masses.stellar"], np.nan)
        _bh_ok  = np.isfinite(BH["bh_mass"]) & np.isfinite(BH["bh_fedd"])
        _mbh_ok = BH["bh_mass"] > 10 ** JET_LOGMBH
        wjet_hist = np.where(_bh_ok, np.where(_mbh_ok,
                             np.clip(np.log10(JET_FEDD / np.clip(BH["bh_fedd"], 1e-12, None)), 0.0, 1.0),
                             0.0), np.nan)
        xcoup_hist = np.where(np.isfinite(wjet_hist) & np.isfinite(fgas_hist),
                              wjet_hist * (fgas_hist < XRAY_FGAS_MAX).astype(float), np.nan)
    n = len(records)
    xstr_q = np.full(n, np.nan)
    for i, r in enumerate(records):
        if not (np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"])):
            continue                                   # no quench event -> stays NaN ('no_event')
        cs = xcoup_hist[_ord, r["col"]].astype(float)
        fin = np.isfinite(cs)
        win = (t_inc >= r["t_sft"]) & (t_inc <= r["t_qt"]) & fin
        if not win.any() and fin.any():
            # quench window narrower than the snapshot spacing -> nearest finite snapshot to SFT
            j = np.where(fin)[0]
            win = np.zeros_like(fin); win[j[np.argmin(np.abs(t_inc[j] - r["t_sft"]))]] = True
        if win.any():
            xstr_q[i] = np.nanmean(cs[win])
    bx = np.isfinite(xstr_q)
    strong = np.zeros(n, bool); weak = np.zeros(n, bool); lo_q = hi_q = np.nan
    if bx.sum() >= 3:
        lo_q, hi_q = np.nanquantile(xstr_q[bx], [1.0 / 3.0, 2.0 / 3.0])
        strong = bx & (xstr_q >= hi_q); weak = bx & (xstr_q <= lo_q)   # §8j-style terciles
    inter = bx & ~strong & ~weak
    no_fb = ~bx
    return dict(xstr_quench=xstr_q, strong=strong, weak=weak, inter=inter, no_fb=no_fb,
                tercile=(lo_q, hi_q))

def agn_class_labels(CO, records):
    """Per-record string label; galaxies without a quench event are 'no_event'."""
    n = len(records)
    has_event = np.array([np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"]) for r in records])
    lab = np.array(["unclassified"] * n, dtype=object)
    if CO is not None:
        lab[CO["no_fb"]] = "no_AGN"
        lab[CO["inter"]] = "intermediate"
        lab[CO["weak"]]  = "weak"
        lab[CO["strong"]] = "strong"
    lab[~has_event] = "no_event"
    return lab

In [ ]:
# ── driver: per anchor -> selection, records, coupling, labels ──
RESULTS = {}
for _zt, A in ANCHORS.items():
    if not os.path.exists(A["hist_path"]):
        print(f"[{A['tag']}] MISSING history -> run BUILD_MULTI_Z on the cluster; skipped")
        continue
    H = load_anchor_history(A)
    m, cuts = selection_mask(H["P"], H["t_cosmic_yr"])
    cols = np.where(m)[0]
    recs = quench_records(H["P"], H["t_cosmic_yr"], H["redshift"], H["galaxy_ids"], cols)
    BH = load_bh(A["bh_path"]) if os.path.exists(A["bh_path"]) else None
    CO = coupling_quench_window(BH, H["P"], recs, H["t_cosmic_yr"]) if BH is not None else None
    labels = agn_class_labels(CO, recs)
    if BH is None:
        print(f"[{A['tag']}] WARNING: no BH history -> AGN split = 'unclassified' (run BUILD_BH)")
    RESULTS[_zt] = dict(A=A, H=H, mask=m, cuts=cuts, cols=cols, records=recs, CO=CO, labels=labels)
    n_ev = int(np.isfinite([r["t_qt"] for r in recs]).sum())
    print(f"[{A['tag']}] snap {A['snap']} (z={A['z']:.3f}): pool={m.size} "
          f"selected={len(cols)} with_event={n_ev} "
          f"classes={dict(zip(*np.unique(labels, return_counts=True))) if len(labels) else {}}")

# Part 3 — Sample statistics & the selection catalog

How many galaxies survive each cut per snapshot, how many have gas at all, and how the AGN classes
populate. **Note:** the pool is the history's build-time pre-selection (massive + passive at the
anchor), not the full galaxy catalog — the funnel starts there. Also writes the per-galaxy
selection table (`powderday_quenched_selection.fits`) that Stages 0–2 read, so the RT stages never
depend on this session's memory.

In [ ]:
# ── funnel table + per-galaxy selection FITS ──
_rows, _sel_rows = [], []
for _zt, R in RESULTS.items():
    A, H, cuts = R["A"], R["H"], R["cuts"]
    ngas0 = H["P"]["ngas"][0]
    n_pool = int(np.isfinite(H["P"]["masses.stellar"][0]).sum())
    lab = R["labels"]
    _rows.append(dict(
        z_target=_zt, snap=A["snap"], z_snap=round(A["z"], 4),
        pool_massive_passive=n_pool,
        with_any_gas=int((ngas0 > 0).sum()),
        gas_gt20=int(cuts["gas>20"].sum()),
        massive=int(cuts["massive"].sum()),
        passive=int(cuts["passive"].sum()),
        star_ge20=int(cuts["star>=20"].sum()),
        selected=len(R["cols"]),
        with_event=int(np.isfinite([r["t_qt"] for r in R["records"]]).sum()),
        strong=int((lab == "strong").sum()), weak=int((lab == "weak").sum()),
        intermediate=int((lab == "intermediate").sum()), no_AGN=int((lab == "no_AGN").sum()),
        no_event=int((lab == "no_event").sum()), unclassified=int((lab == "unclassified").sum()),
    ))
    # per-galaxy rows
    P0 = H["P"]
    for i, (r, l) in enumerate(zip(R["records"], lab)):
        c = r["col"]
        with np.errstate(all="ignore"):
            _ms = float(P0["masses.stellar"][0, c])
            _sf = float(P0["sfr"][0, c])
            xs = R["CO"]["xstr_quench"][i] if R["CO"] is not None else np.nan
        _sel_rows.append(dict(
            snap=int(A["snap"]), z_snap=float(A["z"]), z_target=float(_zt),
            gal_id=int(r["gid"]),
            log_mstar=float(np.log10(_ms)) if _ms > 0 else np.nan,
            ssfr=float(_sf / _ms) if _ms > 0 else np.nan,
            ngas=int(P0["ngas"][0, c]), nstar=int(P0["nstar"][0, c]) if "nstar" in P0 else -1,
            t_sft=r["t_sft"], t_qt=r["t_qt"], tau_q=r["tau_q"],
            tau_q_over_tH=r["tau_q_over_tH"], z_qt=r["z_qt"],
            xstr_quench=float(xs), agn_class=str(l),
        ))

STATS = Table(_rows)
STATS.write(os.path.join(TABLEDIR, "powderday_quenched_stats.fits"), overwrite=True)
STATS.pprint(max_width=-1)

SEL = Table(_sel_rows)
SEL.write(SELECTION_FITS, overwrite=True)
print(f"\nselection table: {len(SEL)} galaxies over {len(np.unique(SEL['snap']))} snapshots "
      f"-> {SELECTION_FITS}")

In [ ]:
# ── figures: selection funnel + gas-particle content + AGN classes ──
_zs   = list(RESULTS.keys())
_tags = [RESULTS[z]["A"]["tag"] for z in _zs]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# funnel per anchor
_steps = ["pool_massive_passive", "with_any_gas", "gas_gt20", "selected", "with_event"]
_slbl  = ["massive+passive", "any gas", "gas>20", "all cuts", "SFT/QT found"]
_x = np.arange(len(_zs)); _w = 0.16
for j, (st, sl) in enumerate(zip(_steps, _slbl)):
    axes[0].bar(_x + (j - 2) * _w, [STATS[st][i] for i in range(len(STATS))], width=_w, label=sl)
axes[0].set_xticks(_x); axes[0].set_xticklabels(_tags)
axes[0].set_ylabel("N galaxies"); axes[0].set_title("selection funnel")
axes[0].legend(fontsize=9)

# gas-particle histograms (pool), with the >20 floor
for z in _zs:
    ng = RESULTS[z]["H"]["P"]["ngas"][0]
    ng = ng[np.isfinite(ng) & (ng > 0)]
    if ng.size:
        axes[1].hist(np.log10(ng), bins=25, histtype="step", lw=2, label=RESULTS[z]["A"]["tag"])
axes[1].axvline(np.log10(NGAS_MIN), color="k", ls=":", label=f"ngas={NGAS_MIN}")
axes[1].set_xlabel("log10 ngas (anchor)"); axes[1].set_ylabel("N")
axes[1].set_title("gas-particle content of the pool"); axes[1].legend(fontsize=9)

# AGN classes among the selected
_classes = ["strong", "intermediate", "weak", "no_AGN", "no_event", "unclassified"]
_bot = np.zeros(len(_zs))
for cl in _classes:
    v = np.array([STATS[cl][i] for i in range(len(STATS))], float)
    axes[2].bar(_x, v, bottom=_bot, label=cl)
    _bot += v
axes[2].set_xticks(_x); axes[2].set_xticklabels(_tags)
axes[2].set_ylabel("N selected"); axes[2].set_title("AGN-coupling classes (quench window)")
axes[2].legend(fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "sample_statistics.png"), dpi=150, bbox_inches="tight")
plt.show()

# Part 4 — Stage 0: extract the per-galaxy particle files (cluster)

One HDF5 per galaxy (gas + stars; gas keeps `Dust_Masses`) under
`hydro_dir_base/snap_NNN/<PREFIX>_snap<NNN>_gal<ID>.h5`. Identical for `dust_on` and `dust_off`
(the dust treatment lives in the parameter master). Reads `SELECTION_FITS`, so it can run in a
fresh session once Part 3 has been executed.

In [ ]:
from simbanator.analysis import extract_particles

SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)
print(f"{len(SNAPS)} sources over snapshots {sorted(set(SNAPS.tolist()))}")

EXTRACT_OVERWRITE = False
EXTRACT_PTYPES    = ("PartType0", "PartType4")   # gas (carries Dust_Masses) + stars

bad_snaps = []
for _snap in np.unique(SNAPS):
    _snap = int(_snap)
    _ids_here = np.unique(IDS[SNAPS == _snap])
    _simfile = sim.get_snapshot_file(_snap)
    print(f"snap {_snap:3d}: extracting {len(_ids_here)} galaxies from {os.path.basename(_simfile)}")
    try:
        _cs = sim.load_catalog(snap=_snap)
        extract_particles(_cs, _simfile, _snap, galaxy_ids=_ids_here, ptypes=EXTRACT_PTYPES,
                          sim_name=sim.name, prefix=PARTICLE_PREFIX,
                          overwrite=EXTRACT_OVERWRITE, verbose=1)
        del _cs
    except (OSError, KeyError) as e:
        print(f"  [SKIP] snap {_snap}: {type(e).__name__}: {str(e).splitlines()[0]}")
        bad_snaps.append((_snap, len(_ids_here)))

print("\nparticle extraction complete ->", hydro_dir_base)
if bad_snaps:
    print(f"{len(bad_snaps)} snapshot(s) unreadable: {bad_snaps} — re-stage those files and re-run.")

# Part 5 — Stage 1: selection HDF5 + Slurm masters (both dust runs)

## ⚠ REQUIRED once per powderday install: the multi-aperture patch

Stock powderday gives the peeled SED a **single infinite aperture**; the parameter masters in this
repo now carry `SED_APERTURE_NAP / SED_APERTURE_MIN_KPC / SED_APERTURE_MAX_KPC`, but powderday must
be taught to read them. On the cluster, locate the peeled-image setup:

```bash
grep -rn "add_peeled_images" $(python -c "import powderday, os; print(os.path.dirname(powderday.__file__))")
```

and immediately after the image-configuration lines (`set_viewing_angles` / `set_track_origin` /
`set_uncertainties`), insert (adapt `image` / `cfg.par` to the local variable names in that file):

```python
# --- multi-aperture SEDs (analize_simba_cgm patch) ---
try:
    from hyperion.util.constants import kpc as _kpc
    _nap = int(getattr(cfg.par, 'SED_APERTURE_NAP', 0))
    if _nap > 0:
        image.set_aperture_range(_nap,
                                 float(cfg.par.SED_APERTURE_MIN_KPC) * _kpc,
                                 float(cfg.par.SED_APERTURE_MAX_KPC) * _kpc)
        image.set_uncertainties(True)   # Monte-Carlo SED errors -> <filter>_err columns
except Exception as _e:
    print('[aperture patch] skipped:', _e)
```

Hyperion **log-spaces** the apertures between min and max: 10→160 kpc with `NAP=5` gives exactly
**10, 20, 40, 80, 160 kpc**. Verify with the Part 6 QC cell after the first galaxy finishes.

Then run this cell, launch `submit_all_snaps.sh` under each run's `powderday_sed_out/`, and come
back to Part 6/7 when the `.rtout.sed` files exist.

In [ ]:
# ── MakeSED handles (constructor only — cheap; Parts 6–7 need just this cell, not the next) ──
from simbanator.sed.makesed import MakeSED

makeseds = {
    key: MakeSED(sim, nnodes=1, model_run_name=cfg['run_tag'],
                 hydro_dir_base=hydro_dir_base, selection_file=selection_file,
                 output_dir=sed_output_dir, run_tag=cfg['run_tag'])
    for key, cfg in RUNS.items()
}
for key, ms in makeseds.items():
    print(f"{key:9s} -> run_tag='{ms.run_tag}', master='{RUNS[key]['paramf']}'")

In [ ]:
# ── write the selection HDF5 + generate the Slurm masters (run once per sample change) ──
SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)

for key, cfg in RUNS.items():
    ms = makeseds[key]
    print(f"\n=== {key} (run_tag='{ms.run_tag}', master='{cfg['paramf']}') ===")
    ms.selection_gals(snaps=SNAPS, galaxyID=IDS)                 # same sources for both runs
    ms.create_master('cluster', 'plist', radius=None,
                     partition='INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL',
                     prefix=PARTICLE_PREFIX, paramf=cfg['paramf'], snaps_to_run=None)

# Part 6 — Aperture QC (run after the first `.rtout.sed` exists)

Confirms the powderday patch took effect **before** burning time on the full extraction:
reads the aperture layout stored in one output file, probes every aperture index, and prints
the expected index → radius mapping.

In [ ]:
from simbanator.sed.makesed import list_sed_apertures, _read_sed

_pat = os.path.join(makeseds['dust_on'].model_dir_base, 'snap_*', 'gal_*', '*.rtout.sed')
_cands = sorted(glob.glob(_pat))
if not _cands:
    raise FileNotFoundError(f"no .rtout.sed yet under {makeseds['dust_on'].model_dir_base} — "
                            "run the RT jobs first")
_probe = _cands[0]
print("probing:", _probe, "\n")

for gname, entry in list_sed_apertures(_probe).items():
    print(f"[{gname}] seds shape = {entry.get('seds_shape')}")
    for k, v in entry.get('seds_attrs', {}).items():
        print(f"    seds.attrs[{k!r}] = {v}")
    for k, v in entry['group_attrs'].items():
        print(f"    group.attrs[{k!r}] = {v}")

print("\nexpected mapping (log-spaced, from the parameter master):")
for i, (r, l) in enumerate(zip(APERTURE_RADII_KPC, APERTURE_LABELS)):
    print(f"  aperture={i} -> {r:7.1f} kpc  ({l})")

n_ok = 0
for i in range(N_AP):
    try:
        wav, flx, unc = _read_sed(_probe, aperture=i, uncertainties=True)
        has_unc = unc is not None and np.isfinite(np.asarray(unc)).any()
        print(f"  aperture={i}: OK  flux shape={np.shape(flx)}  MC uncertainties={'yes' if has_unc else 'NO'}")
        n_ok += 1
    except Exception as e:
        print(f"  aperture={i}: FAILED ({type(e).__name__}: {e})")
assert n_ok == N_AP, (
    f"only {n_ok}/{N_AP} apertures readable — the powderday aperture patch is NOT active "
    "(or N_AP here disagrees with SED_APERTURE_NAP in the parameter master the jobs copied)")
print(f"\nOK — {N_AP} apertures present.")

# Part 7 — Stage 2: per-aperture flux extraction → catalogs

For each dust run × aperture: convolve the SED (and its Monte-Carlo uncertainty) with the filter
set, then join the sample metadata. **One catalog per aperture per dust mode** under
`output/cis25/sed_aperture_catalogs/`, columns: `gal_id, snap, redshift`, sample metadata
(`agn_class, log_mstar, ngas, t_sft, t_qt, tau_q, xstr_quench`, …) and per-filter
`<filter>` / `<filter>_err` fluxes (mJy, rest-frame convolution as in `test_powderday.ipynb`).

In [ ]:
# ── filter set (same as test_powderday: optical->FIR so the dust bump is traced) ──
FACILITIES  = ['HST', 'JWST', 'Spitzer', 'Spitzer', 'Herschel', 'Herschel']
INSTRUMENTS = ['WFC3', 'NIRCam', 'IRAC', 'MIPS', 'PACS', 'SPIRE']

local_filters = {
    '2MASS':   {'J': {'J': REMOTE_HOME + '/2MASS_J.res'}},
    'Johnson': {'V': {'V': REMOTE_HOME + '/maiz-apellaniz_Johnson_V.res'}},
    # separate top-level key required: dict cannot hold two entries under 'Johnson'
    'Johnson2': {'U': {'U': REMOTE_HOME + '/maiz-apellaniz_Johnson_U.res'}},
}

SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)

FLUX_FILES = {}
for key in RUNS:
    ms = makeseds[key]
    for i, label in enumerate(APERTURE_LABELS):
        print(f"\n=== extract: {key} / {label} (aperture index {i}) ===")
        flux_file, xmean_file = ms.extract_flux_batch(
            SNAPS, IDS, FACILITIES, INSTRUMENTS,
            filters=None, local_filters=local_filters, wave_unit='micron', findx=0,
            aperture=i, uncertainties=True,
            outname=f"fluxes_{key}_{label}.fits",
        )
        FLUX_FILES[(key, label)] = flux_file

In [ ]:
# ── final catalogs: fluxes+errors ⨝ sample metadata; one file per (dust mode, aperture) ──
_META = ["gal_id", "snap", "z_snap", "z_target", "agn_class", "xstr_quench",
         "log_mstar", "ngas", "nstar", "ssfr", "t_sft", "t_qt", "tau_q", "tau_q_over_tH"]
SEL = Table.read(SELECTION_FITS)

CATALOGS = {}
for (key, label), ff in FLUX_FILES.items():
    t = Table.read(ff)
    if len(t) == 0:
        print(f"[{key}/{label}] EMPTY flux table — skipped"); continue
    t.rename_column('gal_id_at_snap', 'gal_id')
    cat = join(t, SEL[_META], keys=['snap', 'gal_id'], join_type='left')
    flux_cols = [c for c in t.colnames if c not in ('gal_id', 'snap', 'redshift')]
    cat = cat[['gal_id', 'snap', 'redshift'] + [c for c in _META if c not in ('gal_id', 'snap')]
              + flux_cols]
    out = os.path.join(CATDIR, f"catalog_{key}_{label}.fits")
    cat.meta['APERTURE'] = label
    cat.meta['APIDX'] = APERTURE_LABELS.index(label)
    cat.meta['DUSTRUN'] = key
    cat.write(out, overwrite=True)
    CATALOGS[(key, label)] = out
    n_err = sum(1 for c in cat.colnames if c.endswith('_err'))
    print(f"[{key}/{label}] {len(cat)} galaxies, {n_err} error columns -> {out}")

# ── cross-check: per aperture, dust_on and dust_off must contain the same sources ──
print()
for label in APERTURE_LABELS:
    fon, foff = FLUX_FILES.get(('dust_on', label)), FLUX_FILES.get(('dust_off', label))
    if fon is None or foff is None:
        continue
    t_on, t_off = Table.read(fon), Table.read(foff)
    s_on  = set(zip(np.asarray(t_on['snap'], int), np.asarray(t_on['gal_id_at_snap'], int)))
    s_off = set(zip(np.asarray(t_off['snap'], int), np.asarray(t_off['gal_id_at_snap'], int)))
    status = "OK" if s_on == s_off else f"MISMATCH on={sorted(s_on - s_off)} off={sorted(s_off - s_on)}"
    print(f"{label:>16s}: dust_on={len(s_on)} dust_off={len(s_off)} -> {status}")

# Part 7b — CIGALE input files (one per dust mode × aperture)

Writes `output/cis25/sed_aperture_catalogs/cigale/cigale_{dust_on|dust_off}_{ap…}.fits` in the
exact input format of **CIGALE 2025.0**. All the format/mapping logic lives in
**`simbanator.sed.cigale`** (band names verified against the 2025.0 filter database):

- columns `id` (`snapNNN_galID`), `redshift`, `distance` (Mpc, Planck13 — the same D_L used to
  normalize the fluxes), then per band the flux **in mJy** + its `<band>_err`;
- band names match the CIGALE DB exactly (`jwst.nircam.F200W`, `hst.wfc3.ir.F160W`,
  `spitzer.irac.I1`, `herschel.pacs.green`, `2mass.J`, `generic.johnson.U/V`, …); bands with no
  CIGALE counterpart (grisms, quad filters) are dropped and reported;
- missing fluxes are NaN; make an error negative by hand for upper-limit treatment.

Two deliberate choices:

1. **Observed frame.** CIGALE compares redshifted models to observed photometry, so the
   extraction reruns with `redshift=True` (the Part 7 catalogs stay rest-frame).
2. **Raw MC errors (`err_floor=0`).** CIGALE itself adds `additionalerror` (10 % by default,
   set in Part 7c's `prepare_run`) in quadrature at fit time — a floor here too would be
   double-counted. The Hyperion MC error alone is just RT convergence noise.

In [ ]:
# ── Part 7b: CIGALE 2025.0 input files — observed-frame fluxes+errors, one per (dust mode, aperture) ──
# Format + band mapping live in simbanator.sed.cigale (verified against the 2025.0 filter DB).
# Needs the Part 5 MakeSED handles + the Part 7 filter-set cell in this session.
from simbanator.sed.cigale import write_cigale_input

CIGALE_DIR = os.path.join(CATDIR, "cigale")
os.makedirs(CIGALE_DIR, exist_ok=True)

SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)

CIGALE_FILES = {}
for key in RUNS:
    ms = makeseds[key]
    for i, label in enumerate(APERTURE_LABELS):
        print(f"\n=== CIGALE extract: {key} / {label} (aperture index {i}, observed frame) ===")
        flux_file, _ = ms.extract_flux_batch(
            SNAPS, IDS, FACILITIES, INSTRUMENTS,
            filters=None, local_filters=local_filters, wave_unit='micron', findx=0,
            aperture=i, uncertainties=True,
            redshift=True,                    # CIGALE wants observed-frame fluxes at 'redshift'
            outname=f"cigale_fluxes_{key}_{label}.fits",
        )
        # err_floor=0: CIGALE adds its own 10% 'additionalerror' in quadrature at fit time
        CIGALE_FILES[(key, label)] = write_cigale_input(
            flux_file, os.path.join(CIGALE_DIR, f"cigale_{key}_{label}.fits"),
            err_floor=0.0)

print(f"\n{len(CIGALE_FILES)} CIGALE input files -> {CIGALE_DIR}")

# Part 7c — configure & run CIGALE (**local machine**)

Fully replaces `pcigale init` → edit → `genconf` → edit → `check` → `run`:
`simbanator.sed.cigale.prepare_run` writes a complete, validated `pcigale.ini` + `.spec`
(bands auto-read from the data file), and `run` executes the fit. One run directory per
(dust mode, aperture) under `output/cis25/cigale_runs/`; results in each `…/out/results.fits`.

**Runs only on the local machine** (CIGALE 2025.0 lives in the `cigale-env` conda env; the
cluster has no pcigale). The cell is self-contained — it finds the Part 7b catalogs either in a
local tree or through the gvfs mount, and imports `cigale.py` by file path so none of
simbanator's heavy dependencies (h5py, caesar, …) are needed in `cigale-env`.

**Defaults** (all overridable via `module_params=` / `analysis_params=` in `prepare_run`):
`sfhdelayedbq` (delayed SFH + burst/quench episode — `age_bq`/`r_sfr` map directly onto the
quench window of this sample) + `bc03` (Chabrier, Z = 0.008/0.02/0.05) + `nebular` +
`dustatt_modified_CF00` (Av_ISM 0→2) + `dl2014` + `redshifting`; `additionalerror = 0.1`;
saved variables include `sfh.age_bq`, `sfh.r_sfr`, `stellar.m_star`, `sfh.sfr*`,
`dust.luminosity`. That grid is **126 000 models per redshift** — fine for one anchor,
noticeable for five; trim the grids in `module_params` if runtime matters.

In [ ]:
# ── Part 7c: prepare + run CIGALE on every input file (LOCAL machine, self-contained) ──
import os, glob, importlib.util

PCIGALE_CMD  = os.path.expanduser("~/anaconda3/envs/cigale-env/bin/pcigale")
GVFS_CLUSTER = ("/run/user/1000/gvfs/sftp:host=slurmusrint2.cis.gov.pl,user=glorenzon"
                "/mnt/home/glorenzon/analize_simba_cgm")
REPO_LOCAL   = os.path.expanduser("~/analize_simba_cgm")
PLOT_SEDS    = True    # one best-fit SED figure per object -> <run_dir>/out/<id>_best_model.png

# Part 7b catalogs: prefer a local copy, fall back to the cluster via gvfs
_cands = [os.path.join(REPO_LOCAL, "output", "cis25", "sed_aperture_catalogs", "cigale"),
          os.path.join(GVFS_CLUSTER, "output", "cis25", "sed_aperture_catalogs", "cigale")]
CIGALE_IN = next((d for d in _cands if os.path.isdir(d)), None)
assert CIGALE_IN, f"no Part 7b catalogs found in {_cands} — run Part 7b (cluster) first"
RUN_BASE = os.path.join(REPO_LOCAL, "output", "cis25", "cigale_runs")

# import cigale.py by file path: cigale-env lacks simbanator's heavy deps (h5py, caesar)
try:
    from simbanator.sed import cigale as cg
except ModuleNotFoundError:
    _s = importlib.util.spec_from_file_location(
        "cigale", os.path.join(REPO_LOCAL, "simbanator", "sed", "cigale.py"))
    cg = importlib.util.module_from_spec(_s); _s.loader.exec_module(cg)

CIGALE_RESULTS = {}
for data_file in sorted(glob.glob(os.path.join(CIGALE_IN, "cigale_*.fits"))):
    tag = os.path.basename(data_file)[len("cigale_"):-len(".fits")]   # e.g. dust_on_ap10kpc
    run_dir = os.path.join(RUN_BASE, tag)
    print(f"\n=== {tag} ===")
    cg.prepare_run(run_dir, data_file)   # defaults: sfhdelayedbq+bc03+nebular+CF00+dl2014
    # to shrink the grid, e.g.:
    # cg.prepare_run(run_dir, data_file, module_params={
    #     "sfhdelayedbq": {"tau_main": [1000, 2000], "age_bq": [300, 1000]},
    #     "dustatt_modified_CF00": {"Av_ISM": [0.0, 0.5, 1.0]}})
    CIGALE_RESULTS[tag] = cg.run(run_dir, pcigale_cmd=PCIGALE_CMD, skip_if_done=True)
    if PLOT_SEDS:
        cg.plot_seds(run_dir, format="png",
                     pcigale_plots_cmd=PCIGALE_CMD + "-plots")

print("\nresults:")
for tag, path in CIGALE_RESULTS.items():
    print(f"  {tag}: {path}")

# Run order (cheat sheet)

1. **cluster** — `BUILD_MULTI_Z=True` → run Parts 0–1 (histories); then `BUILD_BH=True` → Part 1
   BH cell. Flip both back to `False` afterwards.
2. Parts 2–3 (selection, AGN split, statistics, `powderday_quenched_selection.fits`) — needs only
   the HDF5s from step 1.
3. **cluster** — Part 4 (Stage 0 particle files), apply the **powderday aperture patch** (Part 5
   markdown), Part 5 cell, then `bash submit_all_snaps.sh` in **both** run trees under
   `output/cis25/sed_quenched_apertures/<run_tag>/powderday_sed_out/`.
4. When `.rtout.sed` files exist: Part 6 QC (must show 5 apertures + MC uncertainties), then
   Part 7 → the per-aperture catalogs, and Part 7b → the CIGALE 2025.0 input files
   (`sed_aperture_catalogs/cigale/`).
5. **local machine** — Part 7c: `simbanator.sed.cigale.prepare_run` + `run` fit every input file
   with CIGALE (`cigale-env`); results in `output/cis25/cigale_runs/<tag>/out/results.fits`.

**Caveats.**
- Aperture radii are hyperion-log-spaced (10/20/40/80/160 kpc), not {10,20,30}; the largest
  aperture doubles as "total" for a plist cutout. `N_AP/AP_MIN_KPC/AP_MAX_KPC` here must match
  `SED_APERTURE_*` in `simbanator/sed/parameters_master*.py` at RT time.
- `<filter>_err` is the Hyperion **Monte-Carlo photon noise** propagated through the filter
  convolution — it is an RT-convergence error, not a mock observational depth. All-NaN error
  columns mean the run stored no uncertainties (patch not applied / `set_uncertainties` missing).
- Fluxes are rest-frame convolved (`redshift=False`, as in `test_powderday.ipynb`); pass
  `redshift=True` in Part 7 for observed-frame photometry (Part 7b does this for CIGALE).
- The dust_off run uses 1 dust-RT photon (`parameters_master-nodust.py`) — some galaxies can
  crash/truncate; the cross-check + `missing_sources_*.txt` make any loss explicit.
- CIGALE error budget: the input files carry raw MC errors; the fit adds `additionalerror = 0.1`
  (10 %) in quadrature via `prepare_run` — change it there, not in Part 7b.